In [25]:
from pyensembl import EnsemblRelease
import pandas as pd

In [26]:
data = EnsemblRelease(110)  # or whichever release you downloaded
transcript = data.transcript_by_id("ENST00000252826")  # TRPM4 canonical

In [4]:
df = pd.read_csv("../data/Q8TD43_peptides.txt", sep="\t", header=None, comment='#')
df.columns = ["peptide", "prev_aa", "next_aa", "length", "start", "end",
              "charges", "col8", "probability", "col10", "col11", "col12",
              "mods", "col14", "col15", "col16", "protein", "accession",
              "protein_name", "gene", "description", "col22", "mapped_proteins"]

In [22]:
def peptide_to_genomic_blocks(aa_start, aa_end, transcript):
    """Returns list of (chrom, start, end, strand) genomic blocks, 1-based inclusive."""
    cds_nt_start = (aa_start - 1) * 3 + 1
    cds_nt_end = aa_end * 3
    
    # coding_sequence_position_ranges: list of (genomic_start, genomic_end) per CDS chunk
    # Ordered 5' → 3' along the transcript (reversed genomic order on - strand)
    cds_ranges = transcript.coding_sequence_position_ranges
    strand = transcript.strand  # "+" or "-"
    
    blocks = []
    cds_consumed = 0
    for gstart, gend in cds_ranges:
        chunk_len = gend - gstart + 1
        chunk_cds_start = cds_consumed + 1
        chunk_cds_end = cds_consumed + chunk_len
        
        # Does this CDS chunk overlap our peptide's CDS range?
        overlap_cds_start = max(cds_nt_start, chunk_cds_start)
        overlap_cds_end = min(cds_nt_end, chunk_cds_end)
        
        if overlap_cds_start <= overlap_cds_end:
            offset_start = overlap_cds_start - chunk_cds_start
            offset_end = overlap_cds_end - chunk_cds_start
            
            if strand == "+":
                block_gstart = gstart + offset_start
                block_gend = gstart + offset_end
            else:
                # On - strand, 5' of CDS = high genomic coordinate
                block_gstart = gend - offset_end
                block_gend = gend - offset_start
            
            blocks.append((transcript.contig, block_gstart, block_gend, strand))
        
        cds_consumed += chunk_len
        if cds_consumed >= cds_nt_end:
            break
    
    return blocks

In [7]:
blocks_per_peptide = [
    peptide_to_genomic_blocks(row["start"], row["end"], transcript)
    for _, row in df.iterrows()
]

In [9]:
def blocks_to_bed12(blocks, name, score=1000):
    # blocks: list of (chrom, start, end, strand), 1-based from Ensembl
    chrom = "chr" + blocks[0][0]  # IGV usually wants "chr" prefix
    strand = "+" if blocks[0][3] == 1 else "-"
    
    # Sort by genomic position (for - strand they come in reverse order from Ensembl)
    blocks_sorted = sorted(blocks, key=lambda b: b[1])
    
    chrom_start = blocks_sorted[0][1] - 1  # BED is 0-based
    chrom_end = blocks_sorted[-1][2]       # BED end is exclusive, Ensembl is inclusive → same number
    
    block_sizes = [b[2] - b[1] + 1 for b in blocks_sorted]
    block_starts = [b[1] - 1 - chrom_start for b in blocks_sorted]
    
    return "\t".join([
        chrom, str(chrom_start), str(chrom_end), name, str(score), strand,
        str(chrom_start), str(chrom_end), "0,0,255",
        str(len(blocks_sorted)),
        ",".join(map(str, block_sizes)) + ",",
        ",".join(map(str, block_starts)) + ","
    ])

In [10]:
with open("trpm4_peptides.bed", "w") as f:
    f.write('track name="TRPM4 peptides" itemRgb=On\n')
    for (_, row), blocks in zip(df.iterrows(), blocks_per_peptide):
        f.write(blocks_to_bed12(blocks, row["peptide"]) + "\n")

In [13]:
import pandas as pd
import re
from io import StringIO

In [12]:
#headers_str = "Peptide	Prev AA	Next AA	Peptide Length	Protein Start	Protein End	Class	Charges	Probability	Qvalue	Spectral Count	Intensity	Assigned Modifications	Observed Modifications	Is Decoy	Is Contaminant	Protein	Protein ID	Entry Name	Gene	Protein Description	Mapped Genes	Mapped Proteins"
#columns = headers_str.split("\t")

In [14]:
columns = [
    "peptide", "prev_aa", "next_aa", "length",
    "start", "end", "class", "charges",
    "probability", "qvalue", "spectral_count", "intensity",
    "assigned_mods", "observed_mods", "is_decoy", "is_contaminant",
    "protein", "protein_id", "entry_name", "gene",
    "description", "mapped_genes", "mapped_proteins"
]

In [15]:
ews = ['A673',
'CHLA10',
'CHLA258',
'CHLA9',
'COH',
'ES1',
'EW1',
'EW16',
'EW18',
'EW22',
'EW23',
'EW24',
'EW3',
'EW7',
'KO_NG3_Clone_B6',
'MHHES1',
'MIC',
'ORS',
'POE',
'RDES',
'SKES1',
'SKNMC',
'STAET1',
'STAET72',
'TC71',
'MSC_EF1',
'ASP14_MinusDox_D7']

nonews = ['CAPAN1',
'HCT116',
'HDMB03',
'IGROV1',
'MP41',
'OVCAR4',
'SHEP',
'SHSY5Y',
'MSC11', 'MSC13FT', 'MSC15FL', 'MSC7BJ', 'V54','ASP14_PlusDox_D7']

In [16]:
def parse_concatenated_tsv(text):
    """Parse a text with multiple ### sample/peptide.tsv ### sections into a dict of DataFrames."""
    # Split on the section header pattern: ### something/peptide.tsv ###
    pattern = r'### (.+?)/peptide\.tsv ###\n'
    parts = re.split(pattern, text)
    # parts[0] is whatever's before the first header (usually empty)
    # then alternates: sample_name, content, sample_name, content, ...
    
    sample_dfs = {}
    for i in range(1, len(parts), 2):
        sample_name = parts[i].strip()
        content = parts[i + 1].strip()
        if not content:
            continue
        df = pd.read_csv(
            StringIO(content),
            sep="\t",
            header=None,
            names=columns,
            engine="python",
        )
        df["sample"] = sample_name
        sample_dfs[sample_name] = df
    
    return sample_dfs

In [17]:
with open("../data/Q8TD43_peptides.txt") as f:
    text = f.read()

sample_dfs = parse_concatenated_tsv(text)

# Concatenate by group
ews_df = pd.concat([sample_dfs[s] for s in ews if s in sample_dfs], ignore_index=True)
nonews_df = pd.concat([sample_dfs[s] for s in nonews if s in sample_dfs], ignore_index=True)

In [18]:
def count_to_red_rgb(count, min_count, max_count, invert=True):
    """
    Map spectral count to red-scale RGB string for BED itemRgb column.
    invert=True: high count → deep red, low count → light pink
    invert=False: high count → light pink, low count → deep red
    """
    if max_count == min_count:
        return "200,0,0"  # mid red as fallback
    
    norm = (count - min_count) / (max_count - min_count)
    if not invert:
        norm = 1 - norm
    
    # High norm → deep red (255, 0, 0)
    # Low norm → light pink (255, 200, 200)
    # Keep R at 255, vary G and B inversely with norm
    gb = int((1 - norm) * 200)  # 0 (deep red) to 200 (light pink)
    return f"255,{gb},{gb}"

def add_rgb_column(df, min_count, max_count, invert=True):
    df = df.copy()
    df["rgb"] = df["spectral_count"].apply(
        lambda c: count_to_red_rgb(c, min_count, max_count, invert)
    )
    return df

In [19]:
all_counts = pd.concat([ews_df["spectral_count"], nonews_df["spectral_count"]])
gmin, gmax = all_counts.min(), all_counts.max()
ews_df = add_rgb_column(ews_df, gmin, gmax)
nonews_df = add_rgb_column(nonews_df, gmin, gmax)

In [20]:
def blocks_to_bed12(blocks, name, score=1000, rgb="0,0,0"):
    # blocks: list of (chrom, start, end, strand), 1-based from Ensembl
    chrom = "chr" + blocks[0][0]  # IGV usually wants "chr" prefix
    strand = "+" if blocks[0][3] == 1 else "-"
    
    # Sort by genomic position (for - strand they come in reverse order from Ensembl)
    blocks_sorted = sorted(blocks, key=lambda b: b[1])
    
    chrom_start = blocks_sorted[0][1] - 1  # BED is 0-based
    chrom_end = blocks_sorted[-1][2]       # BED end is exclusive, Ensembl is inclusive → same number
    
    block_sizes = [b[2] - b[1] + 1 for b in blocks_sorted]
    block_starts = [b[1] - 1 - chrom_start for b in blocks_sorted]
    
    return "\t".join([
        chrom, str(chrom_start), str(chrom_end), name, str(score), strand,
        str(chrom_start), str(chrom_end), rgb,   # ← uses passed rgb
        str(len(blocks_sorted)),
        ",".join(map(str, block_sizes)) + ",",
        ",".join(map(str, block_starts)) + ","
    ])

In [27]:
blocks_per_peptide_ews = [
    peptide_to_genomic_blocks(row["start"], row["end"], transcript)
    for _, row in ews_df.iterrows()
]

In [28]:
with open("trpm4_peptides_ews.bed", "w+") as f:
    f.write('track name="TRPM4 peptides EwS" itemRgb=On\n')
    for (_, row), blocks in zip(ews_df.iterrows(), blocks_per_peptide_ews):
        f.write(blocks_to_bed12(blocks, row["peptide"] , 1000, row['rgb']) + "\n")

In [29]:
blocks_per_peptide_nonews = [
    peptide_to_genomic_blocks(row["start"], row["end"], transcript)
    for _, row in nonews_df.iterrows()
]

with open("trpm4_peptides_nonews.bed", "w+") as f:
    f.write('track name="TRPM4 peptides non EwS" itemRgb=On\n')
    for (_, row), blocks in zip(nonews_df.iterrows(), blocks_per_peptide_nonews):
        f.write(blocks_to_bed12(blocks, row["peptide"] , 1000, row['rgb']) + "\n")

In [30]:
def print_rgb_legend(min_count, max_count, color_fn, invert=True, n_steps=None):
    """Print a legend mapping spectral counts to their RGB values."""
    if n_steps is None:
        n_steps = max_count - min_count + 1
    
    print(f"{'Count':>6}  {'RGB':<15}  Preview")
    print("-" * 40)
    for count in range(min_count, max_count + 1):
        rgb = color_fn(count, min_count, max_count, invert)
        # ANSI escape for terminal color preview (works in most terminals)
        r, g, b = map(int, rgb.split(","))
        preview = f"\033[48;2;{r};{g};{b}m       \033[0m"
        print(f"{count:>6}  {rgb:<15}  {preview}")

print_rgb_legend(gmin, gmax, count_to_red_rgb)

 Count  RGB              Preview
----------------------------------------
     1  255,200,200             
     2  255,180,180             
     3  255,160,160             
     4  255,140,140             
     5  255,120,120             
     6  255,100,100             
     7  255,80,80               
     8  255,60,60               
     9  255,39,39               
    10  255,19,19               
    11  255,0,0                 


In [43]:
class FakeTranscript:
    def __init__(self, contig, strand, coding_sequence_position_ranges,
                 transcript_id="custom", transcript_name="custom",
                 gene_id="custom", biotype="custom", start=None, end=None):
        self.contig = contig
        self.strand = strand
        self.coding_sequence_position_ranges = coding_sequence_position_ranges
        self.transcript_id = transcript_id
        self.transcript_name = transcript_name
        self.gene_id = gene_id
        self.biotype = biotype
        # If start/end not provided, infer from CDS ranges
        if start is None:
            start = min(s for s, _ in coding_sequence_position_ranges)
        if end is None:
            end = max(e for _, e in coding_sequence_position_ranges)
        self.start = start
        self.end = end

In [44]:
dorf_transcript = FakeTranscript(
    contig="19",
    strand="+",
    coding_sequence_position_ranges=[(49211612, 49211851)],  # fill in real coords
    transcript_id="TCONS_00433586_5495_5734",
    transcript_name="TRPM4_dORF",
    gene_id="ENSG00000130529",
    biotype="dORF",
)

In [45]:
with open("../data/TCONS_00433586_5495_5734_peptides.txt") as f:
    text = f.read()

sample_dfs = parse_concatenated_tsv(text)

# Concatenate by group
ews_dorf_df = pd.concat([sample_dfs[s] for s in ews if s in sample_dfs], ignore_index=True)

In [46]:
ews_dorf_df['peptide'].unique()

<ArrowStringArray>
['VLGVTGTTDPSPLTDSSHWGNK', 'TTFGSVILTNHSMPGSSQNQSQPGR', 'AWIPGRYPSGGCR']
Length: 3, dtype: str

In [47]:
ews_dorf_df = add_rgb_column(ews_dorf_df, gmin, gmax)

In [48]:
blocks_per_peptide_ews_dorf = [
    peptide_to_genomic_blocks(row["start"], row["end"], dorf_transcript)
    for _, row in ews_dorf_df.iterrows()
]

In [49]:
with open("trpm4_dorf_peptides_ews.bed", "w+") as f:
    f.write('track name="TCONS_00433586_5495_5734 peptides EwS" itemRgb=On\n')
    for (_, row), blocks in zip(ews_dorf_df.iterrows(), blocks_per_peptide_ews_dorf):
        f.write(blocks_to_bed12(blocks, row["peptide"] , 1000, row['rgb']) + "\n")